# ShopSmart — Minimal RAG demonstration

Retrieval-augmented generation (RAG) supplies catalog evidence to a local language
model before it writes an answer. Here it complements the personalized ML
recommenders with natural-language product discovery and metadata explanations.
It does not change their rankings or validation results. This notebook reuses
`src.rag`; no LLM or network is required for its default execution.

In [1]:
from pathlib import Path
import sys
import json
import os
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.rag import ProductIndex, OllamaClient, answer_question, build_prompt, load_catalog
index = ProductIndex(load_catalog(), ROOT / '.cache/rag/index.joblib')
print('Valid products:', len(index.products), '| Excluded records:', index.excluded_records)
print('Cache reused:', index.cache_hit)


Valid products: 999 | Excluded records: 0
Cache reused: True


## 1. Product documents and retrieval

Titles, categories, features and descriptions remain linked to their parent-ASIN.
Invalid IDs/titles are excluded. TF-IDF word unigrams/bigrams represent the linked catalog documents; retrieval
ranks their cosine similarity to the question.
The local index is cached and rebuilt when source content changes. Scores are
lexical similarity, not ratings or personalized recommendation scores. There
are no arbitrary zero-overlap results.

In [2]:
questions = [
    'Recommend a monitor for office work.',
    'I need wireless headphones for everyday use.',
    'Suggest electronic accessories for a student.',
]
for question in questions:
    result = answer_question(index, question, top_k=3, retrieval_only=True)
    print(question, '|', result.status)
    display(pd.DataFrame([dict(product_id=h.product_id, title=h.title,
                               retrieval_similarity=h.retrieval_score) for h in result.hits]))


Recommend a monitor for office work. | retrieval_only
I need wireless headphones for everyday use. | retrieval_only
Suggest electronic accessories for a student. | retrieval_only


,product_id,title,retrieval_similarity
0,B015WCV70W,HP Pavilion 22cwa 21.5-Inch Full HD 1080p IPS ...,0.096258
1,B00T38OHQ8,Volutz Equilibrium Micro-USB Cable (5-Pack; 10...,0.079750
2,B09NWGVKZ3,"Avantree DG45 USB Bluetooth Adapter for PC, 5....",0.072556


,product_id,title,retrieval_similarity
0,B00LP6CFEC,SoundPEATS Wireless Headphones Stereo Earbuds ...,0.138489
1,B0BXX85J3Y,Avantree Audition Pro 40 hr Bluetooth Over Ear...,0.122628
2,B01EL7VTC0,Bose QuietComfort 35 (Series I) Wireless Headp...,0.113809


,product_id,title,retrieval_similarity
0,B08FVMRW4D,Belkin 7-Outlet Commercial Power Strip Surge P...,0.108863
1,B079ZNSHDN,Tripp Lite 3 Outlet Portable Surge Protector P...,0.108123
2,B0BB6R89VF,"Belkin Surge Protector Power Cube, Power Strip...",0.078899


## 2. Prompt construction and local generation

The prompt contains the question, bounded product documents, IDs and exact
candidate evidence excerpts. A system instruction treats catalog text as data,
requires citations, separates facts from inferred suitability, and forbids
invented specifications, prices, discounts or ranking scores. Ollama receives a
JSON schema; returned IDs and quotes are checked before rendering an answer.

Set `RUN_LIVE = True` only when the Ollama server and a downloaded model are
available. `OLLAMA_MODEL` chooses the model; otherwise the smallest installed
local model is selected. No models are downloaded automatically. A missing
server/model or invalid output produces labelled retrieval-only results.

In [3]:
question = questions[0]
hits = index.search(question, top_k=2)
print(build_prompt(question, hits)[:1800])
RUN_LIVE = False
result = answer_question(index, question, top_k=2,
                         client=OllamaClient(model=os.environ.get('OLLAMA_MODEL')),
                         retrieval_only=not RUN_LIVE)
print('Status:', result.status)
print(result.answer if result.answer else result.reason)


{"mode": "discovery", "question": "Recommend a monitor for office work.", "products": [{"product_id": "B015WCV70W", "title": "HP Pavilion 22cwa 21.5-Inch Full HD 1080p IPS LED Monitor, Tilt, VGA and HDMI (T4Q59AA) - Black", "catalog_document": "Title: HP Pavilion 22cwa 21.5-Inch Full HD 1080p IPS LED Monitor, Tilt, VGA and HDMI (T4Q59AA) - Black\nCategories: Electronics Computers & Accessories Monitors\nFeatures: VGA, HDMI Environmental features: Mercury-free LED backlighting, Arsenic-free monitor glass, and the low halogen design promote energy efficiency Tech Sense Lab headset Headphones (black-white) NOTE: Please refer to the user manual for trouble shooting steps\nDescription: Experience content on a multi-functional monitor with the best specs at the right price. The HP Pavilion 22cwa 21.5” Full HD display with an IPS panel is versatile for gaming with vivid colors and provides 178-degree viewing for the best quality from any position. A VGA cable is included for your convenience 

## 3. Recorded live example

The file below records a real local Ollama call from release verification. Its
status distinguishes generated output from fallback; the notebook does not
pretend that loading a recorded answer performs a new generation.

In [4]:
record_path = ROOT / 'reports/results/rag_live_demo.json'
if record_path.exists():
    recorded = json.loads(record_path.read_text())
    print('Recorded question:', recorded['question'])
    print('Recorded status:', recorded['status'], '| Model:', recorded.get('model'))
    print(recorded.get('answer') or recorded.get('reason'))
else:
    print('No recorded live generation is available.')


Recorded question: Recommend a monitor for office work.
Recorded status: generated | Model: mistral:latest
[B015WCV70W] HP Pavilion 22cwa 21.5-Inch Full HD 1080p IPS LED Monitor, Tilt, VGA and HDMI (T4Q59AA) - Black
Catalog evidence: HP Pavilion 22cwa 21.5-Inch Full HD 1080p IPS LED Monitor, Tilt, VGA and HDMI (T4Q59AA) - Black
Inferred suitability: The HP Pavilion 22cwa monitor is a suitable option for office work due to its Full HD resolution, IPS panel for wide viewing angles, and inclusion of both VGA and HDMI ports. However, the product description does not explicitly state that it is designed specifically for office use.
Missing information: The product documentation does not provide information about the response time, refresh rate, or any specific office-related features such as blue light reduction or ergonomic stand adjustments.


## 4. Explain existing ML recommendations

Mode B looks up the exact IDs returned by the existing recommender, preserving
its order. The LLM sees metadata, not numeric ranking scores or user history.
These are suitability explanations, not causal explanations of KNN internals.
Unknown users inherit the recommender's training-popularity fallback.

In [5]:
bundle_path = ROOT / 'artifacts/recommenders.joblib'
if bundle_path.exists():
    import joblib
    bundle = joblib.load(bundle_path)
    user = bundle['data'].users[0]
    recommendations = bundle['models'][bundle['winner']].recommend(user, 3)
    print('Recommender:', bundle['winner'], '| User:', user)
    display(pd.DataFrame(recommendations, columns=['product_id', 'ml_ranking_score']))
    explanation = answer_question(index, 'Why might these products be suitable?',
                                  product_ids=[product_id for product_id, _ in recommendations],
                                  retrieval_only=True)
    print('Status:', explanation.status, '| Mode:', explanation.mode)
    display(pd.DataFrame([dict(product_id=h.product_id, title=h.title) for h in explanation.hits]))
else:
    print('Run scripts/run_experiments.py to build the optional ML explanation bundle.')


Recommender: Item_V1 | User: AE225JSR3YVYTIDQSVM4P5OIPOCQ
Status: retrieval_only | Mode: explanation


,product_id,ml_ranking_score
0,B00KPRWAX8,0.221670
1,B08SCF2HC2,0.191403
2,B01DLKRLAW,0.181477


,product_id,title
0,B00KPRWAX8,"Intel Core i7-4790K Processor (8M Cache, up to..."
1,B08SCF2HC2,SABRENT 3.5 Inch to x2 SSD / 2.5 Inch Internal...
2,B01DLKRLAW,Corsair CX Series 750 Watt 80 Plus Bronze Cert...


## Limitations

Lexical retrieval can miss synonyms and return adjacent accessories. Exact
quotes and known IDs support grounding but cannot prove free-text inference
quality. Metadata is historical; current availability, prices and Black Friday
discounts are unknown. This extension has functional tests, not an answer-quality
benchmark. Core ML results remain development validation with no independent
final test; the frozen catalog excludes most later positives. See `docs/rag.md`
and `docs/evaluation_protocol.md` for setup and methodological details.